In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import importlib, subprocess, sys


from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [5]:
df = pd.read_csv('spam[1].csv', encoding='latin-1')

In [6]:
df.sample(5)


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
2040,ham,You always make things bigger than they are,NaN,NaN,NaN
1358,ham,If i start sending blackberry torch to nigeria...,NaN,NaN,NaN
4936,ham,G wants to know where the fuck you are,NaN,NaN,NaN
5508,ham,"Machan you go to gym tomorrow, i wil come lat...",NaN,NaN,NaN
1804,ham,The bus leaves at &lt;#&gt;,NaN,NaN,NaN


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   v1          5572 non-null   str  
 1   v2          5572 non-null   str  
 2   Unnamed: 2  50 non-null     str  
 3   Unnamed: 3  12 non-null     str  
 4   Unnamed: 4  6 non-null      str  
dtypes: str(5)
memory usage: 217.8 KB


In [8]:
df = df.dropna(how="any", axis=1)
df.columns = ['target', 'text']

In [11]:

print(f"✅ Data Loaded! Total Rows: {df.shape[0]}")

✅ Data Loaded! Total Rows: 5572


In [12]:
tfidf = TfidfVectorizer(max_features=3000, stop_words='english')
X = tfidf.fit_transform(df['text']).toarray()


In [13]:
le = LabelEncoder()
y = le.fit_transform(df['target'])

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
model = Sequential([
    Dense(512, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.4), # Overfitting se bachne ke liye regularizer
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid') # Binary Output (Spam or Not)
])

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1782243049.414962    3359 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [16]:
model.compile(optimizer='adam', 
              loss='binary_crossentropy', 
              metrics=['accuracy'])


In [17]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [18]:
print("🚀 Training the Heavy AI Model...")
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

🚀 Training the Heavy AI Model...
Epoch 1/10


W0000 00:00:1782243098.660858    3359 cpu_allocator_impl.cc:82] Allocation of 53484000 exceeds 10% of free system memory.


70/70 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.8661 - loss: 0.2972 - val_accuracy: 0.8655 - val_loss: 0.1565
Epoch 2/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9713 - loss: 0.0992 - val_accuracy: 0.9821 - val_loss: 0.0806
Epoch 3/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9946 - loss: 0.0238 - val_accuracy: 0.9839 - val_loss: 0.0745
Epoch 4/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9984 - loss: 0.0068 - val_accuracy: 0.9848 - val_loss: 0.0807
Epoch 5/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9993 - loss: 0.0041 - val_accuracy: 0.9821 - val_loss: 0.0984
Epoch 6/10
70/70 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9987 - loss: 0.0037 - val_accuracy: 0.9839 - val_loss: 0.1002


In [ ]:
predictions = (model.predict(X_test) > 0.5).astype("int32")

print("\n--- CONFUSION MATRIX ---")
print(confusion_matrix(y_test, predictions))

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_test, predictions, target_names=['Ham (Genuine)', 'Spam']))

model.save("advanced_nlp_ann_model.h5")
print("💾 Model Saved Successfully as 'advanced_nlp_ann_model.h5'!")

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step



--- CONFUSION MATRIX ---
[[959   6]
 [ 12 138]]

--- CLASSIFICATION REPORT ---
               precision    recall  f1-score   support

Ham (Genuine)       0.99      0.99      0.99       965
         Spam       0.96      0.92      0.94       150

     accuracy                           0.98      1115
    macro avg       0.97      0.96      0.96      1115
 weighted avg       0.98      0.98      0.98      1115

💾 Model Saved Successfully as 'advanced_nlp_ann_model.h5'!
